In [1]:
import pandas as pd

loans_raw = pd.read_csv("data/loans.csv")
print(loans_raw.shape)
loans_raw.info()

(623, 9)
<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 43.9 KB


In [2]:
print(f"Value count before strip: {loans_raw["genre"].value_counts}")

Value count before strip: <bound method IndexOpsMixin.value_counts of 0              ROMANCE
1                Crime
2              fantasy
3              fantasy
4                CRIME
            ...       
618         children's
619            history
620            fantasy
621           children
622    science fiction
Name: genre, Length: 623, dtype: str>


In [5]:
print(loans_raw["genre"].nunique())
loans_raw["genre"] = loans_raw["genre"].str.strip().str.lower()
loans_raw["genre"] = loans_raw["genre"].str.replace({"children's": "children"})
loans_raw["genre"] = loans_raw["genre"].str.replace({"sci-fi": "science fiction"})
loans_raw["genre"] = loans_raw["genre"].str.replace({"sci fi": "science fiction"})
loans_raw["genre"] = loans_raw["genre"].str.title()
print(loans_raw["genre"].value_counts())

28
genre
Crime              163
Children           135
Fantasy             90
Science Fiction     90
Romance             86
History             59
Name: count, dtype: int64


In [6]:
print("Distinct values:", loans_raw["returned"].nunique())
print(loans_raw["returned"].value_counts(dropna=False))

loans_raw["returned"] = (
    loans_raw["returned"]
    .str.strip()
    .str.lower()
    .map({"yes": True, "no": False})
)

print(loans_raw["returned"].value_counts(dropna=False))

Distinct values: 10
returned
Y      131
Yes    125
YES    124
y      113
yes    100
N        8
NO       7
No       6
n        6
no       3
Name: count, dtype: int64
returned
True     349
NaN      258
False     16
Name: count, dtype: int64


In [7]:
loans_raw["late_fee"] = (
    loans_raw["late_fee"]
    .str.replace("kr", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

print(loans_raw["late_fee"].describe())
print("Total fees:", loans_raw["late_fee"].sum())

count    623.000000
mean      12.323435
std       16.106338
min        0.000000
25%        0.000000
50%        0.000000
75%       25.000000
max       47.500000
Name: late_fee, dtype: float64
Total fees: 7677.5


In [8]:
loans_raw["loan_date"] = pd.to_datetime(loans_raw["loan_date"], errors="coerce")

print("Dates converted to NaT:", loans_raw["loan_date"].isna().sum())
print("First date:", loans_raw["loan_date"].min())
print("Last date:", loans_raw["loan_date"].max())

loans_per_month = loans_raw.groupby(
    loans_raw["loan_date"].dt.to_period("M")
).size()

print(loans_per_month)
print("Monthly total:", loans_per_month.sum())
print("Rows with valid dates:", loans_raw["loan_date"].notna().sum())
print("Total rows:", len(loans_raw))

Dates converted to NaT: 3
First date: 2026-01-05 00:00:00
Last date: 2026-06-27 00:00:00
loan_date
2026-01     96
2026-02    100
2026-03    111
2026-04    106
2026-05    108
2026-06     99
Freq: M, dtype: int64
Monthly total: 620
Rows with valid dates: 620
Total rows: 623


In [9]:
loans_original = pd.read_csv("data/loans.csv")

print("Duplicates in raw table:", loans_original.duplicated().sum())
print("Duplicates after cleaning:", loans_raw.duplicated().sum())

Duplicates in raw table: 12
Duplicates after cleaning: 15


In [10]:
loans_raw = pd.read_csv("data/loans.csv")
loans_clean = loans_raw.drop(columns=["notes"], errors="ignore").copy()

loans_clean["genre"] = (
    loans_clean["genre"]
    .str.strip()
    .str.lower()
    .replace({
        "children's": "children",
        "sci-fi": "science fiction",
        "sci fi": "science fiction",
    })
    .str.title()
)

loans_clean["returned"] = (
    loans_clean["returned"]
    .str.strip()
    .str.lower()
    .map({"yes": True, "no": False})
)

loans_clean["late_fee"] = (
    loans_clean["late_fee"]
    .str.replace("kr", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

loans_clean["loan_date"] = pd.to_datetime(
    loans_clean["loan_date"], errors="coerce"
)

print("Shape:", loans_clean.shape)
loans_clean.to_csv("data/loans_clean.csv", index=False)

Shape: (623, 8)
